<a href="https://colab.research.google.com/github/Harshit-563/LLM_fine_tune_Qwen3-4B-with-QLoRA/blob/main/fine_tune_qwen3_4b_with_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import the libraries


In [ ]:
!pip install -U trl

In [ ]:
from collections import Counter
import json
import os
import re

import torch
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import snapshot_download
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer


## Check the GPU


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A GPU is required for this project")

device = torch.device("cuda")
print(torch.cuda.get_device_name(0))
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


Tesla T4
Memory: 14.6 GB


## Create the project folders


In [ ]:
WORKSPACE = "/tmp/tensortonic"
DATA_DIR = os.path.join(WORKSPACE, "data", "medquad")
MODEL_DIR = os.path.join(WORKSPACE, "models")
MODEL_CACHE = os.path.join(MODEL_DIR, "huggingface")
DATASET_CACHE = os.path.join(WORKSPACE, "cache", "datasets")

os.makedirs(MODEL_CACHE, exist_ok=True)
os.makedirs(DATASET_CACHE, exist_ok=True)

print(WORKSPACE)


/tmp/tensortonic


## Download the base model


In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"
print(f"Downloading {MODEL_ID} to {MODEL_CACHE}", flush=True)
local_model_path = snapshot_download(MODEL_ID, cache_dir=MODEL_CACHE)
print(f"Model files ready at {local_model_path}", flush=True)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Model files ready at /tmp/tensortonic/models/huggingface/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c


## Configure 4-bit quantization


In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    # TODO 1: configure NF4, bfloat16 compute, and double quantization.
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
print(quantization)


BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



## Load the tokenizer and model


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(local_model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    quantization_config=quantization,
    device_map={"": 0},
    dtype=torch.bfloat16,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)
print(f"Loaded {MODEL_ID}")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-4B


## Download MedQuAD


In [ ]:
DATASET_ID = "Hmehdi515/MedQuad"
DATASET_FILES = {
    "train": "https://huggingface.co/datasets/Hmehdi515/MedQuad/resolve/main/medquad_train.csv",
    "validation": "https://huggingface.co/datasets/Hmehdi515/MedQuad/resolve/main/medquad_val.csv",
}

print(f"Downloading {DATASET_ID}", flush=True)
source = load_dataset("csv", data_files=DATASET_FILES, cache_dir=DATASET_CACHE)

os.makedirs(DATA_DIR, exist_ok=True)
raw_train_path = os.path.join(DATA_DIR, "raw_train.jsonl")
raw_validation_path = os.path.join(DATA_DIR, "raw_validation.jsonl")
source["train"].to_json(raw_train_path)
source["validation"].to_json(raw_validation_path)

print(f"Saved {len(source['train'])} rows to {raw_train_path}", flush=True)
print(f"Saved {len(source['validation'])} rows to {raw_validation_path}", flush=True)
print(source)


medquad_train.csv: reconstructing file:   0%|          |  0.00B / 22.0MB            

medquad_train.csv: downloading bytes:           |  0.00B            

medquad_val.csv:   0%|          | 0.00/199k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Creating json from Arrow format:   0%|          | 0/17 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved 16078 rows to /tmp/tensortonic/data/medquad/raw_train.jsonl
Saved 164 rows to /tmp/tensortonic/data/medquad/raw_validation.jsonl
DatasetDict({
    train: Dataset({
        features: ['sample', 'question', 'answer'],
        num_rows: 16078
    })
    validation: Dataset({
        features: ['sample', 'question', 'answer'],
        num_rows: 164
    })
})


## Inspect the dataset


In [ ]:
columns = source["train"].column_names

answer_lengths = [
    len(row["answer"].split())
    for row in source["train"]
]

sorted_lengths = sorted(answer_lengths)
n = len(sorted_lengths)

if n % 2 == 1:
    median_answer_words = sorted_lengths[n // 2]
else:
    median_answer_words = (
        sorted_lengths[n // 2 - 1] + sorted_lengths[n // 2]
    ) / 2

assert {"question", "answer"}.issubset(columns)
assert answer_lengths
assert median_answer_words is not None

print({
    "columns": columns,
    "training_rows": len(source["train"]),
    "median_answer_words": median_answer_words,
})
print(source["train"][0])


{'columns': ['sample', 'question', 'answer'], 'training_rows': 16078, 'median_answer_words': 139.0}
{'sample': 12536, 'question': 'What are the symptoms of Male Breast Cancer ?', 'answer': 'Men with breast cancer usually have lumps that can be felt.Lumps and other signs may be caused by male breast cancer or by other conditions. Check with your doctor if you notice a change in your breasts.'}


## Prepare a clean training split


In [ ]:
def select_examples(split, limit):
    rows = []
    for example in source[split]:
        question = str(example["question"]).strip()
        answer = str(example["answer"]).strip()
        answer_words = len(answer.split())

        # TODO 3: keep non-empty questions with answers between 20 and 160 words.
        keep_example = bool(question) and 20 <= answer_words <= 160
        if keep_example:
            rows.append({"question": question, "answer": answer})
        if len(rows) == limit:
            break
    return Dataset.from_list(rows)

print("Creating a small training set and a separate evaluation set", flush=True)
dataset = DatasetDict({
    "train": select_examples("train", 96).shuffle(seed=42),
    "evaluation": select_examples("validation", 8),
})

print(f"Writing dataset files to {DATA_DIR}", flush=True)
dataset["train"].to_json(os.path.join(DATA_DIR, "train.jsonl"))
dataset["evaluation"].to_json(os.path.join(DATA_DIR, "evaluation.jsonl"))
dataset.save_to_disk(os.path.join(DATA_DIR, "arrow"))

SYSTEM_PROMPT = (
    "Answer the medical question clearly and concisely. When a question "
    "requires personal medical advice, recommend consulting a qualified clinician."
)
print(dataset)
print(dataset["train"][0])


Creating a small training set and a separate evaluation set
Writing dataset files to /tmp/tensortonic/data/medquad


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saving the dataset (0/1 shards):   0%|          | 0/96 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 96
    })
    evaluation: Dataset({
        features: ['question', 'answer'],
        num_rows: 8
    })
})
{'question': "What is (are) Crohn's Disease ?", 'answer': "Crohn's disease is a chronic, or long lasting, disease that causes inflammationirritation or swellingin the gastrointestinal (GI) tract. Most commonly, Crohn's affects the small intestine and the beginning of the large intestine. However, the disease can affect any part of the GI tract, from the mouth to the anus.\n                \nCrohn's disease is a chronic inflammatory disease of the GI tract, called inflammatory bowel disease (IBD). Ulcerative colitis and microscopic colitis are the other common IBDs. More information is provided in the NIDDK health topics, Ulcerative Colitis and Microscopic Colitis: Collagenous Colitis and Lymphocytic Colitis.\n                \nCrohn's disease most often begins gradually and can become w

## Build the generation prompt


In [ ]:
def generate_answer(model, question):
    # Create system and user messages
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    # Render the chat prompt for generation
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,
    )

    assert messages, "Add the prompt messages"
    assert inputs is not None, "Render the prompt with the tokenizer"

    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    prompt_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(
        output[0][prompt_length:],
        skip_special_tokens=True
    ).strip()


## Implement the evaluation metric


In [ ]:
import re
from collections import Counter

def answer_f1(prediction, reference):
    # TODO 5: tokenize both answers and count their overlapping words.
    predicted_tokens = re.findall(r"\b\w+\b", prediction.lower())
    reference_tokens = re.findall(r"\b\w+\b", reference.lower())

    if not predicted_tokens or not reference_tokens:
        return float(predicted_tokens == reference_tokens)

    predicted_counts = Counter(predicted_tokens)
    reference_counts = Counter(reference_tokens)

    overlap = sum((predicted_counts & reference_counts).values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(predicted_tokens)
    recall = overlap / len(reference_tokens)

    return 2 * precision * recall / (precision + recall)


## Measure the base model


In [ ]:
def evaluate_model(model, rows):
    model.eval()
    results = []
    for row in rows:
        prediction = generate_answer(model, row["question"])
        score = answer_f1(prediction, row["answer"])
        results.append({
            "question": row["question"],
            "reference": row["answer"],
            "prediction": prediction,
            "token_f1": round(score * 100, 2),
        })
    return {
        "token_f1": round(sum(row["token_f1"] for row in results) / len(results), 2),
        "examples": len(results),
        "results": results,
    }

baseline = evaluate_model(base_model, dataset["evaluation"])
print(json.dumps({
    "token_f1": baseline["token_f1"],
    "examples": baseline["examples"],
}, indent=2))


{
  "token_f1": 25.96,
  "examples": 8
}


## Format examples as conversations


In [ ]:
def format_example(example):
    # Add system, user, and assistant messages in that order.
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": example["question"],
        },
        {
            "role": "assistant",
            "content": example["answer"],
        },
    ]

    # Render messages as text using the tokenizer chat template.
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

    return {"text": text}

train = dataset["train"].map(format_example)

assert train[0]["text"].strip(), "The formatted training text is empty"
print(train[0]["text"])


Map:   0%|          | 0/96 [00:00<?, ? examples/s]

<|im_start|>system
Answer the medical question clearly and concisely. When a question requires personal medical advice, recommend consulting a qualified clinician.<|im_end|>
<|im_start|>user
What is (are) Crohn's Disease ?<|im_end|>
<|im_start|>assistant
<think>

</think>

Crohn's disease is a chronic, or long lasting, disease that causes inflammationirritation or swellingin the gastrointestinal (GI) tract. Most commonly, Crohn's affects the small intestine and the beginning of the large intestine. However, the disease can affect any part of the GI tract, from the mouth to the anus.
                
Crohn's disease is a chronic inflammatory disease of the GI tract, called inflammatory bowel disease (IBD). Ulcerative colitis and microscopic colitis are the other common IBDs. More information is provided in the NIDDK health topics, Ulcerative Colitis and Microscopic Colitis: Collagenous Colitis and Lymphocytic Colitis.
                
Crohn's disease most often begins gradually and can 

## Complete the QLoRA adapter


In [ ]:
lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

print(f"LoRA rank: {lora.r}")


LoRA rank: 8


## Fine-tune the model


In [ ]:
training_config = SFTConfig(
    output_dir="/tmp/qwen-medquad",
    learning_rate=0.0002,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_length=384,
    logging_steps=1,
    save_strategy="no",
    report_to=[],
    bf16=True,
    gradient_checkpointing=True,
    dataset_text_field="text",
    seed=42,
)

# TODO 8
trainer = SFTTrainer(
    model=base_model,
    args=training_config,
    train_dataset=train,
    peft_config=lora,
    processing_class=tokenizer,
)

train_result = trainer.train()

assert trainer is not None, "Create the trainer"
assert train_result is not None, "Start the training run"

print({"training_loss": round(train_result.training_loss, 5)})


Adding EOS to train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,3.877419
2,3.226990
3,2.566149
4,2.595458
5,2.280828
6,2.230791
7,1.811021
8,1.733819
9,1.737048
10,1.447917


{'training_loss': 1.7388}


## Inspect where the model still fails


In [ ]:
trainer.model.eval()
fine_tuned_result = evaluate_model(trainer.model, dataset["evaluation"])

# Calculate score change
score_change = (
    fine_tuned_result["token_f1"] - baseline["token_f1"]
)

# Keep the two weakest trained-model answers
weakest = sorted(
    fine_tuned_result["results"],
    key=lambda x: x["token_f1"]
)[:2]

print(json.dumps({
    "baseline_token_f1": baseline["token_f1"],
    "fine_tuned_token_f1": fine_tuned_result["token_f1"],
    "score_change": score_change,
    "weakest_answers": weakest,
}, indent=2))


{
  "baseline_token_f1": 25.96,
  "fine_tuned_token_f1": 26.1,
  "score_change": 0.14000000000000057,
  "weakest_answers": [
    {
      "question": "What are the treatments for cystinuria ?",
      "reference": "These resources address the diagnosis or management of cystinuria:  - Genetic Testing Registry: Cystinuria  - MedlinePlus Encyclopedia: Cystinuria  - MedlinePlus Encyclopedia: Cystinuria (image)   These resources from MedlinePlus offer information about the diagnosis and management of various health conditions:  - Diagnostic Tests  - Drug Therapy  - Surgery and Rehabilitation  - Genetic Counseling   - Palliative Care",
      "prediction": "Cystinuria is a genetic disorder that causes the body to excrete excessive amounts of cystine in the urine. Cystine is an amino acid that is normally found in the body. When cystine is excreted in large amounts in the urine, it can form crystals that may lead to kidney stones. Cystinuria is inherited in an autosomal recessive pattern. This m